# Cell cycle phase score from p21 / Rb / pRb and DNA content (OPS0077 NTC)

This notebook runs `scripts/ops0077_cell_cycle_score.py` on the OPS0077 non-targeting-control cells and shows the result. It derives a **circular cell cycle phase score** per cell (`cc_phase`, in [0, 1), where 0 and 1 are the same point) from:

- nuclear **p21, Rb and pRb** intensities (4i), summarised as the phosphorylated fraction of Rb (pRb relative to Rb), p21 and pRb
- **relative DNA content** from DAPI (`dna_rel_R234`, used only where `dna_reliable` is true)
- optionally, per-channel / pairwise **PCs** that track the phase (skipped if the PC file is absent)

**How the score is built.** All axes are robust-z-scored per well. A closed principal curve is fitted through the cells in that space. Phase 0 is anchored at the lowest-DNA part of the loop (G1), and the direction runs forward through the p21 minimum (S) to the DNA maximum (G2/M), then back to G1.

**Input data** (relative to the repository root)

| file | content |
|---|---|
| `data/altair/ops0077_4i/OPS0077_NTC_OrganelleProfiler_pRb_Rb_p21.h5ad` | single-cell 4i features |
| `data/altair/OPS0077_NTC_DAPI_cpmeasure.h5ad` | single-cell DAPI features, including DNA content |
| `reports/ops0077_cell_cycle_clock/per_cell_pca_embeddings.csv.gz` | *(optional)* per-channel / pairwise PCs |

**Outputs** go to `reports/ops0077_cell_cycle_score/`. Run time is a few minutes.

## 1. Requirements

In [ ]:
# Run once if needed (uncomment). Python >= 3.10.
# %pip install anndata numpy pandas scipy scikit-learn umap-learn matplotlib

## 2. Configuration and input checks

In [ ]:
import subprocess, sys, json
from pathlib import Path

# repository root: the folder that contains `scripts/` and `data/`
ROOT = Path("..").resolve() if (Path("..") / "scripts").exists() else Path(".").resolve()
SCRIPT = ROOT / "scripts/ops0077_cell_cycle_score.py"
INPUTS = {
    "4i markers (required)": ROOT / "data/altair/ops0077_4i/OPS0077_NTC_OrganelleProfiler_pRb_Rb_p21.h5ad",
    "DAPI / DNA content (required)": ROOT / "data/altair/OPS0077_NTC_DAPI_cpmeasure.h5ad",
    "per-cell PCs (optional)": ROOT / "reports/ops0077_cell_cycle_clock/per_cell_pca_embeddings.csv.gz",
}
OUT = ROOT / "reports/ops0077_cell_cycle_score"

print("repository root:", ROOT)
for name, path in INPUTS.items():
    print(f"{'OK     ' if path.exists() else 'MISSING'} {name}: {path.relative_to(ROOT)}")
missing = [n for n, p in INPUTS.items() if "required" in n and not p.exists()]
assert not missing, f"Missing required input(s): {missing}"
assert SCRIPT.exists(), f"Script not found: {SCRIPT}"


## 3. Run the analysis
This calls the script and streams its log. It will overwrite the files in `reports/ops0077_cell_cycle_score/`.

In [ ]:
proc = subprocess.Popen([sys.executable, "-W", "ignore", str(SCRIPT), str(ROOT)],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "analysis failed, see the log above"


## 4. Summary

In [ ]:
import pandas as pd
summary = json.load(open(OUT / "summary.json"))
print(json.dumps(summary, indent=2))

res = pd.read_csv(OUT / "cell_cycle_scores.csv.gz", index_col=0)
print(f"\n{res['cc_phase'].notna().sum()} of {len(res)} cells have a cc_phase")
res.head()

**Reading the log.** Along the loop, p21 should be lowest around S, DNA content should peak in G2/M, and pRb and the phosphorylated fraction should rise in late G1. `phase vs DNA: harmonic R2` is the share of DNA-content variance explained by the phase. The `markers-only vs DNA-anchored` agreement shows how much the DNA axis adds: markers alone recover the cycle poorly.

## 5. Figures

In [ ]:
from IPython.display import Image, display
for name, caption in [("umap_overview.png", "UMAP of the fitted axes coloured by phase and by each axis"),
                      ("phase_planes.png", "Pairs of axes coloured by phase"),
                      ("marker_profiles.png", "Marker and DNA profiles around the cycle"),
                      ("score_distributions.png", "Phase by well, and agreement between the fits")]:
    print(caption)
    display(Image(filename=str(OUT / name)))

## 6. Quick look at the score

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(13, 4.5))
ax = fig.add_subplot(1, 2, 1, projection="polar")
th = 2 * np.pi * res["cc_phase"].dropna().values
cnt, edges = np.histogram(th, bins=48, range=(0, 2 * np.pi))
ax.bar(edges[:-1], cnt, width=np.diff(edges), align="edge", color="C4", alpha=.85)
ax.set_theta_zero_location("N"); ax.set_theta_direction(-1)
ax.set_title("cells around the cycle (0 = G1 start, clockwise)")

ax = fig.add_subplot(1, 2, 2)
d = res.dropna(subset=["cc_phase"]).sample(min(15000, res["cc_phase"].notna().sum()), random_state=0)
h = ax.scatter(d["z_dna"], d["z_p21"], c=d["cc_phase"], s=2, cmap="twilight", rasterized=True)
ax.set_xlabel("relative DNA content (z)"); ax.set_ylabel("p21 (z)")
plt.colorbar(h, ax=ax, label="cc_phase"); plt.tight_layout()

## 7. Output files

| file | content |
|---|---|
| `cell_cycle_scores.csv.gz` | one row per cell (see columns below) |
| `summary.json` | counts and key statistics of the run |
| `pc_screening.csv` | PC screening table (only if the PC file is present) |
| `*.png` | figures shown above |

**Columns of `cell_cycle_scores.csv.gz`**

| column | meaning |
|---|---|
| `cc_phase` | final score in [0, 1), circular: from `dna+pcs` if available, else `dna` |
| `cc_phase_source` | which fit produced `cc_phase` |
| `cc_phase_dna` | markers + DNA content |
| `cc_phase_dna_pcs` | markers + DNA content + PCs |
| `cc_phase_markers` | markers only, aligned to the DNA phase. **QC only**, it agrees poorly with the DNA-anchored phase |
| `cc_curve_dist_*` | distance from the fitted cycle curve (large = off-cycle or noisy) |
| `z_phos`, `z_p21`, `z_pRb`, `z_dna` | the per-well robust-z axes used in the fit |
| `cc_phase_angle_rad` | `cc_phase` as an angle in radians |

Cells without a reliable DNA measurement have no `cc_phase`.

**Using the score downstream.** The phase is circular, so 0 and 1 are the same biological state. Use `sin(2π·cc_phase)` and `cos(2π·cc_phase)` instead of the raw 0-1 number when comparing, binning or correlating it with other data.